In [1]:
!pip install -q kagglehub

import os
import glob
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy import stats
from PIL import Image
import kagglehub

# =====================================================================
# 1. DOWNLOAD DATASET & PREPARE SPLITS
# =====================================================================
print("📥 Downloading dataset via kagglehub...")
dataset_path = kagglehub.dataset_download("nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone")

csv_search = glob.glob(f"{dataset_path}/**/*.csv", recursive=True)
df = pd.read_csv(csv_search[0])

all_image_paths = glob.glob(f"{dataset_path}/**/*.jpg", recursive=True)
path_dict = {os.path.basename(p): p for p in all_image_paths}
df['path'] = df['path'].apply(lambda x: path_dict.get(os.path.basename(str(x))))
df = df.dropna(subset=['path'])

# Keep 20% for unseen final testing; remaining 80% is for 3-Fold CV
cv_df, test_df = train_test_split(df, test_size=0.2, stratify=df['Class'], random_state=42)
cv_df = cv_df.reset_index(drop=True)
print(f"✅ Dataset ready! {len(cv_df)} samples allocated for 3-Fold Cross-Validation.")

# =====================================================================
# 2. DATASET CLASS & TRANSFORMS
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CTKidneyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        image = Image.open(img_path).convert('RGB')
        label = int(self.dataframe.iloc[idx]['target'])
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# =====================================================================
# 3. DENSENET121 MODEL LOADER (WITH FROZEN BACKBONE)
# =====================================================================
def build_densenet121(num_classes=4):
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    # Freeze pre-trained feature extractor layers to prevent weight corruption
    for param in model.features.parameters():
        param.requires_grad = False
    # Replace final linear layer
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model

# =====================================================================
# 4. 3-FOLD CROSS-VALIDATION EXECUTION (10 EPOCHS PER FOLD)
# =====================================================================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
num_epochs = 10
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

densenet_results = []

print("\n" + "="*60)
print("🚀 INITIALIZING ARCHITECTURE: DenseNet121 (3-Fold CV)")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(cv_df, cv_df['Class']):
    print(f"\n>>> Starting Fold {fold}/3 for DenseNet121 <<<")
    print("-" * 50)

    fold_train_df = cv_df.iloc[train_idx]
    fold_val_df = cv_df.iloc[val_idx]

    fold_train_dataset = CTKidneyDataset(fold_train_df, transform=train_transforms)
    fold_val_dataset = CTKidneyDataset(fold_val_df, transform=test_val_transforms)

    fold_train_loader = DataLoader(fold_train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize fresh model for each fold
    model = build_densenet121(num_classes=4).to(device)
    # Optimize only trainable parameters
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, weight_decay=1e-4)

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        train_loss, train_corrects, train_size = 0.0, 0, 0
        for inputs, labels in fold_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_corrects += torch.sum(preds == labels.data)
            train_size += inputs.size(0)

        epoch_train_loss = train_loss / train_size
        epoch_train_acc = train_corrects.double().item() / train_size

        # Validation Phase
        model.eval()
        val_loss, val_corrects, val_size = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in fold_val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_size += inputs.size(0)

        epoch_val_loss = val_loss / val_size
        epoch_val_acc = val_corrects.double().item() / val_size

        print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | "
              f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.4f}")

        densenet_results.append({
            "Model": "DenseNet121",
            "Fold": fold,
            "Epoch": epoch + 1,
            "Train_Loss": epoch_train_loss,
            "Train_Acc": epoch_train_acc,
            "Val_Loss": epoch_val_loss,
            "Val_Acc": epoch_val_acc
        })

    fold += 1

# =====================================================================
# 5. CALCULATE STATISTICAL METRICS (MEAN, STD DEV, P-VALUE)
# =====================================================================
df_densenet = pd.DataFrame(densenet_results)

# Paired t-test across all 30 epoch data points (3 folds x 10 epochs)
t_stat, p_value = stats.ttest_rel(df_densenet['Train_Loss'], df_densenet['Val_Loss'])

print("\n" + "📊"*20)
print("DENSENET121 3-FOLD CV STATISTICAL METRICS")
print("📊"*20)
print(f"Model Name          : DenseNet121")
print(f"Mean Val Accuracy   : {df_densenet['Val_Acc'].mean()*100:.2f}%")
print(f"Std Dev Val Acc     : {df_densenet['Val_Acc'].std():.4f}")
print(f"Mean Val Loss       : {df_densenet['Val_Loss'].mean():.4f}")
print(f"Std Dev Val Loss    : {df_densenet['Val_Loss'].std():.4f}")
print(f"p-value (Train/Val) : {p_value:.4e}")

# Save results to CSV
df_densenet.to_csv("densenet121_3fold_results.csv", index=False)
print("\n✅ Results saved to 'densenet121_3fold_results.csv'!")

📥 Downloading dataset via kagglehub...
Using Colab cache for faster access to the 'ct-kidney-dataset-normal-cyst-tumor-and-stone' dataset.
✅ Dataset ready! 9956 samples allocated for 3-Fold Cross-Validation.

🚀 INITIALIZING ARCHITECTURE: DenseNet121 (3-Fold CV)

>>> Starting Fold 1/3 for DenseNet121 <<<
--------------------------------------------------
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 164MB/s]


Epoch 01/10 | Train Loss: 0.8395 - Train Acc: 0.6866 | Val Loss: 0.6535 - Val Acc: 0.7987
Epoch 02/10 | Train Loss: 0.5905 - Train Acc: 0.7880 | Val Loss: 0.5592 - Val Acc: 0.8243
Epoch 03/10 | Train Loss: 0.5147 - Train Acc: 0.8154 | Val Loss: 0.4855 - Val Acc: 0.8397
Epoch 04/10 | Train Loss: 0.4630 - Train Acc: 0.8319 | Val Loss: 0.4546 - Val Acc: 0.8506
Epoch 05/10 | Train Loss: 0.4429 - Train Acc: 0.8419 | Val Loss: 0.4071 - Val Acc: 0.8608
Epoch 06/10 | Train Loss: 0.4260 - Train Acc: 0.8450 | Val Loss: 0.4054 - Val Acc: 0.8620
Epoch 07/10 | Train Loss: 0.3905 - Train Acc: 0.8617 | Val Loss: 0.3876 - Val Acc: 0.8650
Epoch 08/10 | Train Loss: 0.3966 - Train Acc: 0.8600 | Val Loss: 0.3788 - Val Acc: 0.8665
Epoch 09/10 | Train Loss: 0.3761 - Train Acc: 0.8664 | Val Loss: 0.4143 - Val Acc: 0.8409
Epoch 10/10 | Train Loss: 0.3572 - Train Acc: 0.8707 | Val Loss: 0.3578 - Val Acc: 0.8843

>>> Starting Fold 2/3 for DenseNet121 <<<
--------------------------------------------------
Epoch 

In [2]:
!pip install -q kagglehub

import os
import glob
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy import stats
from PIL import Image
import kagglehub

# =====================================================================
# 1. DOWNLOAD DATASET & PREPARE SPLITS
# =====================================================================
print("📥 Downloading dataset via kagglehub...")
dataset_path = kagglehub.dataset_download("nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone")

csv_search = glob.glob(f"{dataset_path}/**/*.csv", recursive=True)
df = pd.read_csv(csv_search[0])

all_image_paths = glob.glob(f"{dataset_path}/**/*.jpg", recursive=True)
path_dict = {os.path.basename(p): p for p in all_image_paths}
df['path'] = df['path'].apply(lambda x: path_dict.get(os.path.basename(str(x))))
df = df.dropna(subset=['path'])

# Keep 20% for unseen final testing; remaining 80% is for 3-Fold CV
cv_df, test_df = train_test_split(df, test_size=0.2, stratify=df['Class'], random_state=42)
cv_df = cv_df.reset_index(drop=True)
print(f"✅ Dataset ready! {len(cv_df)} samples allocated for 3-Fold Cross-Validation.")

# =====================================================================
# 2. DATASET CLASS & TRANSFORMS
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CTKidneyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        image = Image.open(img_path).convert('RGB')
        label = int(self.dataframe.iloc[idx]['target'])
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# =====================================================================
# 3. VIT BASE / 16 MODEL LOADER (WITH FROZEN TRANSFORMER BACKBONE)
# =====================================================================
def build_vit_base(num_classes=4):
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
    # Freeze pre-trained transformer backbone
    for param in model.parameters():
        param.requires_grad = False
    # Replace classification head (automatically gets requires_grad=True)
    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model

# =====================================================================
# 4. 3-FOLD CROSS-VALIDATION EXECUTION (10 EPOCHS PER FOLD)
# =====================================================================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
num_epochs = 10
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

vit_results = []

print("\n" + "="*60)
print("🚀 INITIALIZING ARCHITECTURE: ViT Base / 16 (3-Fold CV)")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(cv_df, cv_df['Class']):
    print(f"\n>>> Starting Fold {fold}/3 for ViT Base / 16 <<<")
    print("-" * 50)

    fold_train_df = cv_df.iloc[train_idx]
    fold_val_df = cv_df.iloc[val_idx]

    fold_train_dataset = CTKidneyDataset(fold_train_df, transform=train_transforms)
    fold_val_dataset = CTKidneyDataset(fold_val_df, transform=test_val_transforms)

    fold_train_loader = DataLoader(fold_train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize fresh model for each fold
    model = build_vit_base(num_classes=4).to(device)
    # Optimize only trainable parameters
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, weight_decay=1e-4)

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        train_loss, train_corrects, train_size = 0.0, 0, 0
        for inputs, labels in fold_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_corrects += torch.sum(preds == labels.data)
            train_size += inputs.size(0)

        epoch_train_loss = train_loss / train_size
        epoch_train_acc = train_corrects.double().item() / train_size

        # Validation Phase
        model.eval()
        val_loss, val_corrects, val_size = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in fold_val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_size += inputs.size(0)

        epoch_val_loss = val_loss / val_size
        epoch_val_acc = val_corrects.double().item() / val_size

        print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | "
              f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.4f}")

        vit_results.append({
            "Model": "ViT_Base_16",
            "Fold": fold,
            "Epoch": epoch + 1,
            "Train_Loss": epoch_train_loss,
            "Train_Acc": epoch_train_acc,
            "Val_Loss": epoch_val_loss,
            "Val_Acc": epoch_val_acc
        })

    fold += 1

# =====================================================================
# 5. CALCULATE STATISTICAL METRICS (MEAN, STD DEV, P-VALUE)
# =====================================================================
df_vit = pd.DataFrame(vit_results)

# Paired t-test across all 30 epoch data points (3 folds x 10 epochs)
t_stat, p_value = stats.ttest_rel(df_vit['Train_Loss'], df_vit['Val_Loss'])

print("\n" + "📊"*20)
print("ViT BASE / 16 3-FOLD CV STATISTICAL METRICS")
print("📊"*20)
print(f"Model Name          : ViT Base / 16")
print(f"Mean Val Accuracy   : {df_vit['Val_Acc'].mean()*100:.2f}%")
print(f"Std Dev Val Acc     : {df_vit['Val_Acc'].std():.4f}")
print(f"Mean Val Loss       : {df_vit['Val_Loss'].mean():.4f}")
print(f"Std Dev Val Loss    : {df_vit['Val_Loss'].std():.4f}")
print(f"p-value (Train/Val) : {p_value:.4e}")

# Save results to CSV
df_vit.to_csv("vit_base_3fold_results.csv", index=False)
print("\n✅ Results saved to 'vit_base_3fold_results.csv'!")

📥 Downloading dataset via kagglehub...
Using Colab cache for faster access to the 'ct-kidney-dataset-normal-cyst-tumor-and-stone' dataset.
✅ Dataset ready! 9956 samples allocated for 3-Fold Cross-Validation.

🚀 INITIALIZING ARCHITECTURE: ViT Base / 16 (3-Fold CV)

>>> Starting Fold 1/3 for ViT Base / 16 <<<
--------------------------------------------------
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:05<00:00, 61.2MB/s]


Epoch 01/10 | Train Loss: 0.6319 - Train Acc: 0.7812 | Val Loss: 0.4368 - Val Acc: 0.8575
Epoch 02/10 | Train Loss: 0.3569 - Train Acc: 0.8881 | Val Loss: 0.3163 - Val Acc: 0.9060
Epoch 03/10 | Train Loss: 0.2798 - Train Acc: 0.9161 | Val Loss: 0.2660 - Val Acc: 0.9238
Epoch 04/10 | Train Loss: 0.2315 - Train Acc: 0.9325 | Val Loss: 0.2322 - Val Acc: 0.9337
Epoch 05/10 | Train Loss: 0.2102 - Train Acc: 0.9373 | Val Loss: 0.2073 - Val Acc: 0.9443
Epoch 06/10 | Train Loss: 0.1838 - Train Acc: 0.9471 | Val Loss: 0.1964 - Val Acc: 0.9403
Epoch 07/10 | Train Loss: 0.1668 - Train Acc: 0.9533 | Val Loss: 0.1755 - Val Acc: 0.9506
Epoch 08/10 | Train Loss: 0.1593 - Train Acc: 0.9543 | Val Loss: 0.1781 - Val Acc: 0.9473
Epoch 09/10 | Train Loss: 0.1419 - Train Acc: 0.9622 | Val Loss: 0.1613 - Val Acc: 0.9524
Epoch 10/10 | Train Loss: 0.1341 - Train Acc: 0.9635 | Val Loss: 0.1581 - Val Acc: 0.9515

>>> Starting Fold 2/3 for ViT Base / 16 <<<
--------------------------------------------------
Epoc